# Workflow PRODUZIONE - fit Meridian + allocazione budget

Da usare **dopo** aver preparato i dati in locale con `app_ingestion.py`.
Carichi qui i 4 CSV puliti (`media`, `demand`, `seasonality`, `outcome`),
giri il fit Meridian sulla GPU, fai l'allocazione del budget e scarichi i
risultati.

**Prima di iniziare:** Runtime -> Cambia tipo di runtime -> GPU (T4) -> Salva.

Ordine: setup -> carica i 4 CSV -> fit -> allocazione -> scarica.

## 1. Setup: codice e dipendenze

In [ ]:
!git clone https://github.com/Giacomod2001/Tesi-MMM.git
%cd Tesi-MMM
!pip install -q -r pipeline/requirements.txt

## 2. Carica i 4 CSV puliti

Esegui la cella: comparira' il pulsante "Scegli file". Seleziona i 4 CSV
prodotti dall'app in locale (cartella `MMM_dati_puliti` sul Desktop):
`media.csv`, `demand.csv`, `seasonality.csv`, `outcome.csv`.

In [ ]:
import os
from google.colab import files

CANON = 'pipeline/data/canonical'
os.makedirs(CANON, exist_ok=True)
print('Seleziona: media.csv, demand.csv, seasonality.csv, outcome.csv')
caricati = files.upload()
for nome in caricati:
    os.replace(nome, os.path.join(CANON, nome))
presenti = sorted(os.listdir(CANON))
print('File in canonical:', presenti)
attesi = {'media.csv', 'demand.csv', 'seasonality.csv', 'outcome.csv'}
mancanti = attesi - set(presenti)
assert not mancanti, f'Mancano questi file: {mancanti}'
print('OK: tutti e 4 i file canonici sono pronti.')

## 3. Fit Meridian (GPU, ~20-40 min)

Per una prova rapida solo meccanica (stime NON utilizzabili) aggiungi
`--smoke` al comando.

In [ ]:
!python -m pipeline.model.run

## 4. Allocazione trimestrale del budget

Imposta il budget reale e i vincoli min/max per canale (in EUR) e la data
di inizio del trimestre. L'esempio sotto va adattato ai numeri di Randstad.

In [ ]:
!python -m pipeline.allocator.run --budget 450000 \
    --min linkedin=50000 --max meta=250000 --quarter-start 2026-07-06

### (opzionale) Carica il cruscotto come Foglio Google

Crea `dashboard.xlsx` come **Foglio Google** nel tuo Drive e stampa il link (usa il login Google di Colab: ti chiede l'autorizzazione).

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

drive = build('drive', 'v3')
meta = {'name': 'MMM Dashboard', 'mimeType': 'application/vnd.google-apps.spreadsheet'}
media = MediaFileUpload('pipeline/data/output/dashboard.xlsx',
                        mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
f = drive.files().create(body=meta, media_body=media, fields='id,webViewLink').execute()
print('Foglio Google creato nel tuo Drive:')
print(f['webViewLink'])

## 5. Scarica i risultati

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('produzione', 'zip', 'pipeline/data/output')
files.download('produzione.zip')